# Eval Walkthrough

Pick a form, optionally override strategies, run `run_form()`, inspect metrics. Per-field metrics are consolidated into `<OUT_BASE>/all_forms_metrics.xlsx` (one sheet per form) and the side-by-side comparison workbook lands in `<OUT_BASE>/comparison_sheets/`.


In [8]:
# ── Boilerplate — imports + selection (run once per kernel) ──
import sys, os
sys.path.insert(0, os.path.abspath('.'))

import pandas as pd
from IPython.display import display
from dotenv import load_dotenv
load_dotenv()

from config.field_config import FieldSpec, STRATEGY_LLM_JUDGE, STRATEGY_SKIP, STRATEGY_EXCLUDE
from forms import FORM_REGISTRY                    # oral cancer — 5 forms
from forms.antibiotic import ANTIBIOTIC_REGISTRY   # antibiotic prophylaxis — 4 forms
from forms.periodontitis import PERIODONTITIS_REGISTRY  # periodontitis — 5 forms

GT_DATA_DIR  = "/home/ubuntu/evistream/eval/gt sheets"
AI_BASE_DIR  = "/home/ubuntu/evistream/eval/ai sheets"
OUTPUTS_ROOT = "/home/ubuntu/evistream/eval/outputs"

# ── Pick ALL THREE coordinates yourself (re-run this cell after changing any) ──
VARIANT = "full_studies"   # experiment: full_studies | desc_only | staged | table | user_study
DATASET = "periodontitis"  # corpus:     oral_cancer | antibiotic | periodontitis
MODEL   = "claude"         # model/source: claude | gemini | gpt | claude_code

# Input and output BOTH route to <variant>/<corpus>/<model>/ — they mirror exactly:
#   input : ai sheets/<VARIANT>/<DATASET>/<MODEL>/<form-file>.csv
#   output: outputs/<VARIANT>/<DATASET>/<MODEL>/{all_forms_metrics.xlsx, comparison_sheets/}
# Because the model lives in the path, picking MODEL=gemini reads gemini's input AND
# writes gemini's output (no mismatch). AI_SHEET below overrides the input file only.
AI_DATA_DIR = f"{AI_BASE_DIR}/{VARIANT}/{DATASET}/{MODEL}"
OUT_BASE    = f"{OUTPUTS_ROOT}/{VARIANT}/{DATASET}/{MODEL}"

DATASETS = {
    "oral_cancer":   {"registry": FORM_REGISTRY,          "aligned_gt": f"{GT_DATA_DIR}/aligned_forms_v2.xlsx",      "gt_key_col": "Paper",    "gt_skiprows": None, "canonical_form": "study_characteristics"},
    "antibiotic":    {"registry": ANTIBIOTIC_REGISTRY,    "aligned_gt": f"{GT_DATA_DIR}/antibiotic_prophylaxis.xlsx", "gt_key_col": "study_id", "gt_skiprows": [1],  "canonical_form": "abx_study_char"},
    "periodontitis": {"registry": PERIODONTITIS_REGISTRY, "aligned_gt": f"{GT_DATA_DIR}/periodontitis.xlsx",          "gt_key_col": "study_id", "gt_skiprows": [1],  "canonical_form": "perio_study_char"},
}

DS              = DATASETS[DATASET]
REGISTRY        = DS["registry"]
ALIGNED_GT_PATH = DS["aligned_gt"]

# run_form() consolidates per-field metrics into <OUT_BASE>/all_forms_metrics.xlsx
# (one sheet per form). The side-by-side comparison workbook is written by the
# save_comparison cell into <OUT_BASE>/comparison_sheets/. Study-level match
# tables are an internal cache (a per-process temp dir) — set EVAL_MAPPINGS_DIR
# only if you want to persist/inspect them.
os.environ["EVAL_METRICS_DIR"] = OUT_BASE

print(f"Experiment : {VARIANT}")
print(f"Corpus     : {DATASET}    Model: {MODEL}")
print(f"Forms      : {list(REGISTRY) if REGISTRY else '(registry not defined yet)'}")
print(f"Input dir  : {AI_DATA_DIR}/  {'✓' if os.path.isdir(AI_DATA_DIR) else '✗ MISSING'}")
print(f"GT file    : {ALIGNED_GT_PATH}  {'✓' if os.path.exists(ALIGNED_GT_PATH) else '✗ MISSING'}")
print(f"Output dir : {OUT_BASE}/")

Experiment : full_studies
Corpus     : periodontitis    Model: claude
Forms      : ['perio_study_char', 'perio_patient_pop', 'perio_interventions', 'perio_outcomes', 'perio_risk_of_bias']
Input dir  : /home/ubuntu/evistream/eval/ai sheets/full_studies/periodontitis/claude/  ✓
GT file    : /home/ubuntu/evistream/eval/gt sheets/periodontitis.xlsx  ✓
Output dir : /home/ubuntu/evistream/eval/outputs/full_studies/periodontitis/claude/


In [9]:
# ── Clear the LLM-judge comparison cache ─────────────────────────────
# Run this ONCE to force every field comparison to be re-judged from
# scratch (e.g. after changing the judge prompt or a comparison strategy).
# Clears BOTH the on-disk cache file and the module's in-memory dict —
# clearing only the file doesn't work, since the module re-flushes memory.
# (Study-level match tables in manual_mappings/ are a SEPARATE cache;
#  delete those files or pass force_rematch=True to refresh them.)
import os, sys
sys.path.insert(0, os.path.abspath('.'))
import core.llm_judge as _lj

_n = len(_lj._CACHE)
_lj._CACHE.clear()                       # reset in-memory cache
if os.path.exists(_lj._CACHE_PATH):
    os.remove(_lj._CACHE_PATH)           # delete on-disk cache
print(f'🧹 Cleared LLM-judge cache: {_n} judgments removed → {_lj._CACHE_PATH}')


🧹 Cleared LLM-judge cache: 1 judgments removed → /home/ubuntu/evistream/eval/forms/../core/../cache/llm_judgments.json


In [10]:
# ── User choices — change these per run ──
FORM     = list(REGISTRY)[3]         # pick any key printed by the boilerplate cell
AI_SHEET = '/home/ubuntu/evistream/eval/ai sheets/full_studies/periodontitis/claude/form_continuous_outcomes_long.csv'                      # override the input CSV (None = auto:
                                     #   AI_DATA_DIR/<basename of the form's CFG ai_filename>,
                                     #   i.e. <VARIANT>/<DATASET>/<MODEL>/form_<form>_long.csv).
                                     # Set explicitly for variants with non-standard filenames
                                     #   (staged uses <form>_single_long.csv) or a one-off file.
LLM_OVERRIDE_FIELDS = []             # [] | "all" | ["field1", "field2"]
USE_LLM  = True
INCLUDE_SKIPPED = True               # True → flip skipped fields to llm_judge so they get compared
                                       #         (auto, when both AI and GT have the column)

# ── Build cfg from your choices ──
cfg = dict(REGISTRY[FORM])
# auto-resolve the input under the chosen VARIANT/DATASET/MODEL dir (basename of the
# form's CFG path), unless AI_SHEET overrides it
ai_path = AI_SHEET if AI_SHEET else os.path.join(AI_DATA_DIR, os.path.basename(cfg["ai_filename"]))

# Identifier columns that shouldn't be scored even if INCLUDE_SKIPPED is on
_IDENTIFIER_FIELDS = {"Paper", "study_id", "refid", "authors_last_name"}

if INCLUDE_SKIPPED:
    # Need to know which columns exist in AI + aligned GT, so load shapes quickly
    import pandas as _pd
    _ai_cols = set(_pd.read_csv(ai_path, nrows=0).columns) if cfg["ai_format"] == "csv" \
               else set(_pd.json_normalize(__import__("json").load(open(ai_path))).columns)
    _gt_cols = set(_pd.read_excel(ALIGNED_GT_PATH, sheet_name=cfg["gt_aligned_sheet"], nrows=0).columns)

    promoted = []
    new_fields = []
    for f in cfg["fields"]:
        if (f.strategy == STRATEGY_SKIP
                and f.ai_col not in _IDENTIFIER_FIELDS
                and f.ai_col in _ai_cols
                and f.ai_col in _gt_cols):
            new_fields.append(FieldSpec(ai_col=f.ai_col, strategy=STRATEGY_LLM_JUDGE,
                                        tolerance=f.tolerance, vocab=f.vocab))
            promoted.append(f.ai_col)
        else:
            new_fields.append(f)
    cfg["fields"] = new_fields
    if promoted:
        print(f"↑ Promoted {len(promoted)} skipped fields to llm_judge: {promoted}")

if LLM_OVERRIDE_FIELDS:
    override_set = None if LLM_OVERRIDE_FIELDS == "all" else set(LLM_OVERRIDE_FIELDS)
    cfg["fields"] = [
        FieldSpec(ai_col=f.ai_col, strategy=STRATEGY_LLM_JUDGE,
                  tolerance=f.tolerance, vocab=f.vocab)
        if f.strategy not in (STRATEGY_SKIP, STRATEGY_EXCLUDE)
           and (override_set is None or f.ai_col in override_set)
        else f
        for f in cfg["fields"]
    ]
    print("⚡ LLM override applied")

print(f"Experiment : {VARIANT}   Corpus: {DATASET}   Model: {MODEL}")
print(f"Form    : {FORM}")
print(f"AI file : {ai_path}  {'✓' if os.path.exists(ai_path) else '✗ MISSING'}")
print(f"GT file : {ALIGNED_GT_PATH}  {'✓' if os.path.exists(ALIGNED_GT_PATH) else '✗ MISSING'}")

↑ Promoted 1 skipped fields to llm_judge: ['subgroup']
Experiment : full_studies   Corpus: periodontitis   Model: claude
Form    : perio_outcomes
AI file : /home/ubuntu/evistream/eval/ai sheets/full_studies/periodontitis/claude/form_continuous_outcomes_long.csv  ✓
GT file : /home/ubuntu/evistream/eval/gt sheets/periodontitis.xlsx  ✓


In [11]:
# Sanity-check strategies — shows 3 sample AI values per field
from core.loader import load_ai

_ai_sample = load_ai(ai_path, cfg["ai_format"])

rows = []
for f in cfg["fields"]:
    vals = _ai_sample[f.ai_col].dropna().astype(str).head(3).tolist() if f.ai_col in _ai_sample.columns else []
    rows.append({
        "Field":    f.ai_col,
        "Strategy": f.strategy,
        "Vocab":    f.vocab or "",
        "Tol":      f.tolerance if f.tolerance else "",
        "Sample 1": (vals[0][:60] if len(vals) > 0 else "—"),
        "Sample 2": (vals[1][:60] if len(vals) > 1 else "—"),
        "Sample 3": (vals[2][:60] if len(vals) > 2 else "—"),
    })

display(pd.DataFrame(rows))


,Field,Strategy,Vocab,Tol,Sample 1,Sample 2,Sample 3
0,outcome_type,exact_normalize,,,HbA1c,PPD,BOP
1,timepoint,exact_normalize,,,3-4_months,3-4_months,3-4_months
2,mean_arm1,numeric_tolerance,,0.1,6.47,3.23,11.66
3,sd_arm1,numeric_tolerance,,0.1,0.34,0.37,8.86
4,n_arm1,numeric_exact,,,20,20,20
5,mean_arm2,numeric_tolerance,,0.1,6.98,3.62,22.66
6,sd_arm2,numeric_tolerance,,0.1,0.51,0.64,13.62
7,n_arm2,numeric_exact,,,20,20,20
8,subgroup,llm_judge,,,SI_vs_usual_care,SI_vs_usual_care,SI_vs_usual_care
9,effect_measure,skip,,,—,—,—


In [12]:
from forms.base_form import run_form

result = run_form(
    form_name=FORM,
    fields=cfg["fields"],
    ai_path=ai_path,
    ai_format=cfg["ai_format"],
    gt_section=cfg["gt_section"],
    level2=cfg.get("level2", False),
    level2_cfg=cfg if cfg.get("level2") else None,
    use_llm=USE_LLM,
    extra_col_keywords=cfg.get("extra_col_keywords"),
    gt_aligned_sheet=cfg.get("gt_aligned_sheet"),
    aligned_gt_path=ALIGNED_GT_PATH,
    gt_key_col=DS["gt_key_col"],
    gt_skiprows=DS["gt_skiprows"],
    canonical_form=DS["canonical_form"],
)



  Running form: perio_outcomes
  [matcher] Loaded existing match table from /tmp/evistream_match_1izu0ujf/perio_outcomes_matches.csv
    [match_subrecords] Bukleta 2018: 5 AI × 5 GT → 5 matched
    [match_subrecords] Chen 2012: 20 AI × 10 GT → 10 matched
    [match_subrecords] D'Aiuto 2018: 8 AI × 8 GT → 8 matched
    [match_subrecords] Das 2019: 10 AI × 10 GT → 10 matched
    [match_subrecords] El-Makaky 2020: 5 AI × 5 GT → 5 matched
    [match_subrecords] Engebretson 2013: 12 AI × 12 GT → 12 matched
    [match_subrecords] Gay 2014: 8 AI × 1 GT → 1 matched
    [match_subrecords] Jones 2007: 1 AI × 1 GT → 1 matched
    [match_subrecords] Kapellas 2017: 3 AI × 2 GT → 2 matched
    [match_subrecords] Katagiri 2009: 6 AI × 6 GT → 6 matched
    [match_subrecords] Kaur 2015: 12 AI × 12 GT → 12 matched
    [match_subrecords] Kiran 2005: 6 AI × 6 GT → 6 matched
    [match_subrecords] Koromantzos 2011: 6 AI × 4 GT → 4 matched
    [match_subrecords] Lee 2020: 3 AI × 3 GT → 3 matched
    [match

In [7]:
# Per-field metrics — color-coded
stats_df = result["stats_df"]

if stats_df.empty:
    print("⚠  No field results — comparison loop produced no scored rows.")
else:
    cols = {
        "field":            "Field",
        "n_compared":       "Compared",
        "n_both_nr":        "Both NR",
        "n_gt_nr_ai_value": "GT=NR, AI extracted",
        "pct_agreement":    "% Agreement",
        "tp":               "TP",
        "fp":               "FP",
        "fn":               "FN",
        "macro_precision":  "Precision",
        "macro_recall":     "Recall",
        "macro_f1":         "F1",
        "fn_false_nr":      "AI missed (NR)",
        "strategy":         "Strategy",
    }
    available = [c for c in cols if c in stats_df.columns]
    renamed = stats_df[available].rename(columns=cols)

    def _color_pct(v):
        try: v = float(v)
        except (TypeError, ValueError): return ""
        if v >= 90: return "background-color: #d4edda"
        if v >= 70: return "background-color: #fff3cd"
        return "background-color: #f8d7da"
    def _color_rate(v):
        try: v = float(v)
        except (TypeError, ValueError): return ""
        if v >= 0.85: return "background-color: #d4edda"
        if v >= 0.65: return "background-color: #fff3cd"
        return "background-color: #f8d7da"

    styled = renamed.style.format({
        "% Agreement": "{:.1f}",
        "Precision":   "{:.3f}",
        "Recall":      "{:.3f}",
        "F1":          "{:.3f}",
    }, na_rep="—")
    for col, fn in [("% Agreement", _color_pct), ("Recall", _color_rate),
                    ("Precision", _color_rate), ("F1", _color_rate)]:
        if col in renamed.columns:
            styled = styled.map(fn, subset=[col])
    display(styled)


,Field,Compared,Both NR,"GT=NR, AI extracted",% Agreement,TP,FP,FN,Precision,Recall,F1,AI missed (NR),Strategy
0,outcome_type,150,0,0,96.0,144,6,6,0.960,0.960,0.960,0,exact_normalize
1,timepoint,150,0,0,99.3,149,1,1,0.993,0.993,0.993,0,exact_normalize
2,mean_arm1,150,0,0,72.7,109,23,41,0.826,0.727,0.773,18,numeric_tolerance
3,sd_arm1,150,0,0,73.3,110,14,40,0.887,0.733,0.803,26,numeric_tolerance
4,n_arm1,150,0,0,78.7,118,25,32,0.825,0.787,0.806,7,numeric_exact
5,mean_arm2,150,0,0,72.7,109,20,41,0.845,0.727,0.782,21,numeric_tolerance
6,sd_arm2,150,0,0,71.3,107,14,43,0.884,0.713,0.789,29,numeric_tolerance
7,n_arm2,150,0,0,66.0,99,43,51,0.697,0.660,0.678,8,numeric_exact
8,subgroup,150,0,0,96.7,145,5,5,0.967,0.967,0.967,0,llm_judge


In [ ]:
# Consolidated metrics → outputs/<variant>/<dataset>/<model>/all_forms_metrics.xlsx
# (one workbook, one sheet per form — re-running a form replaces only its sheet,
#  and each sheet now carries the full Agreement-Stats columns: kappa, bootstrap
#  CIs, over/under-extraction breakdowns, MAE).
#
# NOTE: run_form() already wrote this sheet (via reports.metrics_writer), using
# EVAL_METRICS_DIR from the boilerplate cell. The call below is idempotent —
# keep it if you tweaked result["stats_df"] by hand; otherwise it's a no-op repeat.
from reports.metrics_writer import append_form_metrics

out_path = append_form_metrics(FORM, result["stats_df"])
print(f"✓ all_forms_metrics.xlsx updated — sheet '{FORM[:31]}' at {out_path}")

In [ ]:
# Save side-by-side comparison workbook → outputs/<project>/<model>/comparison_sheets/<form>_comparison.xlsx
# Layout per sheet: row 1 header (Study | field1 AI | field1 GT | field2 AI | field2 GT | ...),
#                   row 2 strategy sub-header, row 3+ data. Color-coded by MATCH_* status.
import openpyxl
from openpyxl.styles import PatternFill, Font, Alignment, Border, Side
from openpyxl.utils import get_column_letter

COMPARISON_DIR = f"{OUT_BASE}/comparison_sheets"
os.makedirs(COMPARISON_DIR, exist_ok=True)
out_path = os.path.join(COMPARISON_DIR, f"{FORM}_comparison.xlsx")

# Palette
F_STUDY  = PatternFill("solid", fgColor="F8F9FA")
F_AI_HDR = PatternFill("solid", fgColor="E8A84A")
F_GT_HDR = PatternFill("solid", fgColor="5EBB8A")
F_MATCH  = PatternFill("solid", fgColor="B7EDD0")
F_MISS   = PatternFill("solid", fgColor="F7B8B8")
F_NA     = PatternFill("solid", fgColor="E8E8E8")
HDR_FT   = Font(bold=True, color="FFFFFF")
WRAP     = Alignment(wrap_text=True, vertical="top")
_THIN    = Side(border_style="thin",   color="BDBDBD")
_THICK   = Side(border_style="medium", color="6C6C6C")
B_AI     = Border(left=_THICK, right=_THIN,  top=_THIN, bottom=_THIN)   # left = pair start
B_GT     = Border(left=_THIN,  right=_THICK, top=_THIN, bottom=_THIN)   # right = pair end
B_STUDY  = Border(left=_THICK, right=_THICK, top=_THIN, bottom=_THIN)

comparison_df = result["comparison_df"]
scored = [f for f in cfg["fields"]
          if f.strategy not in ("skip", "exclude") and f"AI_{f.ai_col}" in comparison_df.columns]
strat_by_field = {f.ai_col: f.strategy for f in scored}

def _write_sheet(wb, name, df, fields):
    ws = wb.create_sheet(name)
    # Row 1 — header
    h1 = ws.cell(1, 1, "Study")
    h1.fill = F_STUDY; h1.font = Font(bold=True); h1.border = B_STUDY
    col = 2
    for f_col in fields:
        a = ws.cell(1, col, f_col); a.fill = F_AI_HDR; a.font = HDR_FT; a.border = B_AI
        g = ws.cell(1, col + 1);    g.fill = F_GT_HDR;                  g.border = B_GT
        col += 2
    # Row 2 — strategy sub-header
    h2 = ws.cell(2, 1, "Strategy →")
    h2.fill = F_STUDY; h2.font = Font(italic=True); h2.border = B_STUDY
    col = 2
    for f_col in fields:
        s = strat_by_field.get(f_col, "")
        a = ws.cell(2, col,     f"AI  ({s})"); a.fill = F_AI_HDR; a.font = HDR_FT; a.border = B_AI
        g = ws.cell(2, col + 1, f"GT  ({s})"); g.fill = F_GT_HDR; g.font = HDR_FT; g.border = B_GT
        col += 2
    # Data rows
    for r_idx, (_, row) in enumerate(df.iterrows(), start=3):
        study = row.get("AI_study") or row.get("GT_study") or ""
        sc = ws.cell(r_idx, 1, study); sc.fill = F_STUDY; sc.border = B_STUDY
        col = 2
        for f_col in fields:
            ai_v = row.get(f"AI_{f_col}", "")
            gt_v = row.get(f"GT_{f_col}", "")
            m    = row.get(f"MATCH_{f_col}", "")
            fill = F_MATCH if m == "YES" else F_MISS if m == "NO" else F_NA
            a = ws.cell(r_idx, col,     ai_v); a.fill = fill; a.alignment = WRAP; a.border = B_AI
            g = ws.cell(r_idx, col + 1, gt_v); g.fill = fill; g.alignment = WRAP; g.border = B_GT
            col += 2
    # Widths + freeze
    ws.column_dimensions["A"].width = 26
    for c in range(2, 2 + 2 * len(fields)):
        ws.column_dimensions[get_column_letter(c)].width = 30
    ws.freeze_panes = "B3"
    ws.row_dimensions[1].height = 28
    ws.row_dimensions[2].height = 22

scored_cols = [f.ai_col for f in scored]

# Build workbook
wb = openpyxl.Workbook()
wb.remove(wb.active)

# Sheet 1: All Fields
_write_sheet(wb, "All Fields", comparison_df, scored_cols)

# Sheet 2: Has Mismatch
match_cols = [f"MATCH_{c}" for c in scored_cols if f"MATCH_{c}" in comparison_df.columns]
mismatch_mask = comparison_df[match_cols].eq("NO").any(axis=1) if match_cols else \
                pd.Series([False] * len(comparison_df))
mismatch_df = comparison_df[mismatch_mask]
_write_sheet(wb, "Has Mismatch", mismatch_df, scored_cols)

# Per-field sheets
for f_col in scored_cols:
    safe_name = f_col[:31]
    _write_sheet(wb, safe_name, comparison_df, [f_col])

wb.save(out_path)
print(f"✓ Saved: {out_path}")
print(f"  Sheets: All Fields ({len(comparison_df)} rows), "
      f"Has Mismatch ({len(mismatch_df)} rows), "
      f"+ {len(scored_cols)} per-field sheets")


In [ ]:
# Inspect raw comparison + discrepancy tables
display(result["comparison_df"].head(20))
print(f"\n{len(result['discrepancy_df'])} discrepancies:")
display(result["discrepancy_df"].head(30))


---
## Appendix — Comparison Strategy Reference

| Strategy | How it works | Typical fields |
|---|---|---|
| `exact_normalize` | Lowercase + strip punctuation + synonym map → exact match | Closed categoricals: `study_setting_transformed`, `funding_category`, `positivity_threshold_transformed` |
| `numeric_exact` | Parse both as numbers, equal if identical | Counts: `year_of_study`, `tp`, `fp`, `n_total_patients` |
| `numeric_tolerance` | Parse both as numbers, match if `|ai − gt| ≤ tolerance` | Rounded values: `pct_female` (±1%), `age_central_tendency_value` (±0.5) |
| `date_range_parse` | Extract start/end years from free text, compare year pairs | `study_period` ("Aug 2017 – Oct 2019" vs "2017–2019") |
| `set_compare_terms` | Parse both into canonical term sets via vocab, compute overlap | Multi-value fields: `site_target_condition`, `patient_population_categories`, `severity_target_condition` |
| `parse_percent` | Extract % value from free text, compare with tolerance | `reported_prevalence` ("Prevalence was 38.5%" vs "38.2%") |
| `llm_judge` | Claude reads both values and decides match/no-match on semantic meaning | Free text: `blinding_rs_examiners`, `method_of_patient_selection`, `ses_population` |

### When to switch strategy
- **`exact_normalize` → `llm_judge`**: when mismatches are paraphrases of the same meaning (e.g. `sequence_of_tests`, `time_interval_rs_it`)
- **`numeric_exact` → `numeric_tolerance`**: when rounding differences cause false failures
- **`exact_normalize` → `set_compare_terms`**: when the field can have multiple values (e.g. `country` for multi-country studies)
- **`llm_judge` → `exact_normalize`**: when the field is actually a closed category and LLM is introducing inconsistency


## NR Policy

How "not reported" values are treated in scoring:

| Scenario | AI | GT | Match result | Counted in F1? | Counted in % agreement? |
|---|---|---|---|---|---|
| Both NR | NR | NR | NA | No — excluded | Yes (agreement) |
| AI missed | NR | real value | NO | Yes — FN | No |
| AI over-extracted | real value | NR | NA (FP) | Yes — FP | No |
| Real match | value | same value | YES | Yes — TP | Yes |
| Real mismatch | value A | value B | NO | Yes — FP+FN | No |

**Key assumption:** NR in GT means the reviewer verified the field is absent from the paper.
If GT NR sometimes means "reviewer skipped", precision is underestimated.
